Installation des bibliothéques 

In [1]:
!pip install numpy pandas scikit-learn tensorflow


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Importation des bibliothéques

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

print("Python OK")
print("TensorFlow version:", tf.__version__)

Python OK
TensorFlow version: 2.20.0


Chargement des 3 fichiers CSV, tel que chaque fichier devient un tableau et chaque ligne représente une mesure IMU (ax, ay, az).

In [3]:
off = pd.read_csv("../dataset/off.csv")
normal = pd.read_csv("../dataset/normal.csv")
dys = pd.read_csv("../dataset/dysfunction.csv")

print("off:", off.shape)
print("normal:", normal.shape)
print("dysfunction:", dys.shape)

off.head()


off: (6100, 5)
normal: (6102, 5)
dysfunction: (6101, 5)


,label,t_ms,ax,ay,az
0,off,4269095,0.000732,-0.013916,1.004761
1,off,4269115,-0.001587,-0.015137,1.004639
2,off,4269135,0.000977,-0.014893,1.005249
3,off,4269155,0.001099,-0.016357,1.003418
4,off,4269175,-0.000610,-0.012573,1.004395


Ajout des colonne : label, tel que 0 = off, 1 = normal, 2 = dysfunction.

Concaténation des données tel que :

pd.concat empile les tableaux les uns sous les autres.

ignore_index=True remet un index propre 0,1,2,3…

On garde seulement les colonnes utiles.

In [4]:
off["label"] = 0
normal["label"] = 1
dys["label"] = 2
data = pd.concat([off, normal, dys], ignore_index=True)

# On garde seulement les colonnes utiles
data = data[["ax", "ay", "az", "label"]]

data.head(), data["label"].value_counts()

(         ax        ay        az  label
 0  0.000732 -0.013916  1.004761      0
 1 -0.001587 -0.015137  1.004639      0
 2  0.000977 -0.014893  1.005249      0
 3  0.001099 -0.016357  1.003418      0
 4 -0.000610 -0.012573  1.004395      0,
 label
 1    6102
 2    6101
 0    6100
 Name: count, dtype: int64)

On crée les fenêtres car une vibration n’est pas reconnaissable sur une seule ligne.
donc on donne au modèle un morceau du signal, par ex. 1 seconde.

In [5]:
# Paramètres des fenêtres
WINDOW_SIZE = 50   # 1 seconde à 50 Hz
STEP = 25          # avance de 0.5 seconde

X_windows = []
y_windows = []
# On fait ça séparement par classe pour éviter de mélanger les diff états dans une même fenêtre
def make_windows(df, label, window_size=50, step=25):
    Xw, yw = [], []
    arr = df[["ax", "ay", "az"]].to_numpy(dtype=np.float32)
    for start in range(0, len(arr) - window_size + 1, step):
        window = arr[start:start+window_size]
        Xw.append(window)
        yw.append(label)
    return Xw, yw
    
X0, y0 = make_windows(off, 0, WINDOW_SIZE, STEP)
X1, y1 = make_windows(normal, 1, WINDOW_SIZE, STEP)
X2, y2 = make_windows(dys, 2, WINDOW_SIZE, STEP)

X_windows = np.array(X0 + X1 + X2, dtype=np.float32)
y_windows = np.array(y0 + y1 + y2, dtype=np.int64)

print("X shape (samples, time, features):", X_windows.shape)
print("y shape:", y_windows.shape)


X shape (samples, time, features): (729, 50, 3)
y shape: (729,)


Séparation des fenêtres tel que : on garde 80% pour apprendre, 20% pour tester.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_windows, y_windows, test_size=0.2, random_state=42, stratify=y_windows
)
print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

Train: (583, 50, 3) (583,)
Test : (146, 50, 3) (146,)


Création du modèle 

In [11]:
num_classes = 3

model = tf.keras.Sequential([
    tf.keras.layers.Input(batch_shape=(1, WINDOW_SIZE, 3)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(num_classes, activation="softmax"),
])

# Compilation du modèle

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# Entraînement du modèle

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32
)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (1, 150)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (1, 32)                │         4,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (1, 16)                │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (1, 3)                 │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,411 (21.14 KB)

 Trainable params: 5,411 (21.14 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.3326 - loss: 1.1179 - val_accuracy: 0.2906 - val_loss: 1.1032
Epoch 2/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4034 - loss: 1.0764 - val_accuracy: 0.3675 - val_loss: 1.0691
Epoch 3/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4571 - loss: 1.0559 - val_accuracy: 0.3162 - val_loss: 1.0648
Epoch 4/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5300 - loss: 1.0340 - val_accuracy: 0.8291 - val_loss: 1.0263
Epoch 5/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6910 - loss: 1.0156 - val_accuracy: 0.8462 - val_loss: 1.0176
Epoch 6/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8219 - loss: 0.9859 - val_accuracy: 0.5812 - val_loss: 1.0076
Epoch 7/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7747 - loss: 0.9643 - val_accuracy: 0.8803 - val_loss: 0.9571
Epoch 8/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8433 - loss: 0.9292 - val_accuracy: 0.8205 - val_loss

Test du modèle (évaluation)
On mesure la performance sur de données non vues.

In [12]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test accuracy:", test_acc)

y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(y_test, y_pred, target_names=["off", "normal", "dysfunction"]))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Test accuracy: 0.8287671208381653
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
              precision    recall  f1-score   support

         off       0.86      1.00      0.92        48
      normal       0.74      1.00      0.85        49
 dysfunction       1.00      0.49      0.66        49

    accuracy                           0.83       146
   macro avg       0.87      0.83      0.81       146
weighted avg       0.87      0.83      0.81       146

Confusion matrix:
 [[48  0  0]
 [ 0 49  0]
 [ 8 17 24]]


Export en tensorflow lite(.tflite)

In [14]:
# Conversion en TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
tf.lite.experimental.Analyzer.analyze(model_content=tflite_model)

# Sauvegarde dans le dossier models/
import os
os.makedirs("../models", exist_ok=True)

with open("../models/vibration_model.tflite", "wb") as f:
    f.write(tflite_model)

print("Saved: ../models/vibration_model.tflite")


INFO:tensorflow:Assets written to: C:\Users\sarah\AppData\Local\Temp\tmpg2uiarmj\assets


INFO:tensorflow:Assets written to: C:\Users\sarah\AppData\Local\Temp\tmpg2uiarmj\assets


Saved artifact at 'C:\Users\sarah\AppData\Local\Temp\tmpg2uiarmj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 50, 3), dtype=tf.float32, name='keras_tensor_5')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)
Captures:
  2517850515664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2517850514512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2517846766416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2517850516240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2517850514320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2517850516432: TensorSpec(shape=(), dtype=tf.resource, name=None)
=== TFLite ModelAnalyzer ===

Your TFLite model has '1' subgraph(s). In the subgraph description below,
T# represents the Tensor numbers. For example, in Subgraph#0, the RESHAPE op takes
tensor #0 and tensor #7 as input and produces tensor #8 as output.

Subgraph#0 main(T#0) -> [T#12]
  O